# Late Delivery & Revenue-at-Risk Analysis — Olist Brazilian E-Commerce

**Question:** What's actually causing late deliveries on Olist, and how much repeat-purchase revenue do they cost?

**Data:** [Olist Brazilian E-Commerce dataset](https://www.kaggle.com/olistbr/brazilian-ecommerce) — ~99K orders, Sep 2016–Aug 2018, 9 relational tables. Loaded via DuckDB.

This notebook runs the full pipeline end-to-end: load → baseline late rate → causes (geography vs. product category) → satisfaction impact → repeat-purchase cohort → revenue-at-risk estimate → exports for the Tableau dashboard. Each analysis step also has a standalone, commented `.sql` file in `/sql` for reference.


## Setup: load the 9 CSVs into DuckDB

In [1]:
import duckdb
import pandas as pd
import json
from pathlib import Path

DATA_DIR = Path("../data")  # place the 9 Olist CSVs here (see README for source)
EXPORT_DIR = Path("../data_exports")
EXPORT_DIR.mkdir(exist_ok=True)

con = duckdb.connect(":memory:")

tables = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for name, fname in tables.items():
    con.execute(f"""CREATE OR REPLACE TABLE {name} AS
                 SELECT * FROM read_csv_auto('{DATA_DIR / fname}')""")

for name in tables:
    n = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name:22s} {n:>8,} rows")


orders                   99,441 rows
order_items             112,650 rows
order_payments          103,886 rows
order_reviews            99,224 rows
customers                99,441 rows
sellers                   3,095 rows
products                 32,951 rows
geolocation            1,000,163 rows
category_translation         71 rows


## 1. Baseline late-delivery rate

"Late" = `order_delivered_customer_date > order_estimated_delivery_date`, scoped to `order_status = 'delivered'` only.

In [2]:
baseline = con.execute("""
    SELECT
        CASE WHEN order_delivered_customer_date > order_estimated_delivery_date
             THEN 'late' ELSE 'on_time' END AS delivery_flag,
        COUNT(*) AS n_orders,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_delivered
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
    GROUP BY 1
""").fetchdf()
baseline


,delivery_flag,n_orders,pct_of_delivered
0,on_time,88644,91.89
1,late,7826,8.11


## 2. What's causing lateness? Geography, not product

Two checks: does the customer's state predict lateness, and does crossing a state line (seller → customer) matter?

In [3]:
late_by_state = con.execute("""
    WITH base AS (
        SELECT o.order_id, c.customer_state,
            CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END AS is_late
        FROM orders o JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    )
    SELECT customer_state, COUNT(*) AS n_orders, ROUND(100.0*AVG(is_late),2) AS pct_late
    FROM base GROUP BY 1 ORDER BY n_orders DESC
""").fetchdf()

# ~1.3% of orders have items from more than one seller. Use the order's first item
# (order_item_id = 1) as the representative seller so "route" is single-valued per order.
late_by_route = con.execute("""
    WITH base AS (
        SELECT o.order_id, c.customer_state, s.seller_state,
            CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END AS is_late
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id AND oi.order_item_id = 1
        JOIN sellers s ON oi.seller_id = s.seller_id
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    )
    SELECT CASE WHEN customer_state = seller_state THEN 'same_state' ELSE 'different_state' END AS route,
           COUNT(*) AS n_orders, ROUND(100.0*AVG(is_late),2) AS pct_late
    FROM base GROUP BY 1
""").fetchdf()

late_by_state.to_csv(EXPORT_DIR / "state_late_rates.csv", index=False)
print(late_by_state.head(15).to_string())
print()
print(late_by_route.to_string())


   customer_state  n_orders  pct_late
0              SP     40494      5.89
1              RJ     12350     13.47
2              MG     11354      5.61
3              RS      5344      7.15
4              PR      4923      5.00
5              SC      3546      9.76
6              BA      3256     14.04
7              DF      2080      7.07
8              ES      1995     12.23
9              GO      1957      8.18
10             PE      1593     10.80
11             CE      1279     15.32
12             PA       946     12.37
13             MT       886      6.77
14             MA       717     19.67

             route  n_orders  pct_late
0       same_state     34701      6.06
1  different_state     61769      9.27


**Control check:** does product category predict lateness instead? If it did, we'd expect a wide spread. We don't see one — this rules out "seller/product quality" as the driver and points at logistics/geography.

In [4]:
late_by_category = con.execute("""
    WITH base AS (
        SELECT DISTINCT o.order_id, p.product_category_name,
            CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END AS is_late
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        JOIN products p ON oi.product_id = p.product_id
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    )
    SELECT product_category_name, COUNT(*) AS n_orders, ROUND(100.0*AVG(is_late),2) AS pct_late
    FROM base GROUP BY 1 ORDER BY n_orders DESC LIMIT 10
""").fetchdf()

late_by_category.to_csv(EXPORT_DIR / "category_late_rates.csv", index=False)
late_by_category


,product_category_name,n_orders,pct_late
0,cama_mesa_banho,9272,8.75
1,beleza_saude,8647,8.96
2,esporte_lazer,7529,7.76
3,informatica_acessorios,6529,7.70
4,moveis_decoracao,6307,8.48
5,utilidades_domesticas,5743,6.95
6,relogios_presentes,5493,8.52
7,telefonia,4093,8.53
8,automotivo,3809,8.61
9,brinquedos,3803,7.52


## 3. Satisfaction impact: review score by delivery outcome

In [5]:
review_impact = con.execute("""
    WITH base AS (
        SELECT o.order_id,
            CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'late' ELSE 'on_time' END AS delivery_flag
        FROM orders o
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    )
    SELECT b.delivery_flag, COUNT(*) AS n_orders,
           ROUND(AVG(r.review_score),2) AS avg_review_score,
           ROUND(100.0*SUM(CASE WHEN r.review_score<=2 THEN 1 ELSE 0 END)/COUNT(*),2) AS pct_1_2_star
    FROM base b JOIN order_reviews r ON b.order_id = r.order_id
    GROUP BY 1
""").fetchdf()

review_impact.to_csv(EXPORT_DIR / "review_score_by_delivery.csv", index=False)
review_impact


,delivery_flag,n_orders,avg_review_score,pct_1_2_star
0,on_time,88653,4.29,9.23
1,late,7700,2.57,54.03


## 4. Repeat-purchase cohort

Uses `customer_unique_id` (not `customer_id`, which is unique per order and would make every customer look like a one-time buyer). Tags each customer's *first* delivered order late/on-time, then checks whether they ever ordered again.

In [6]:
cohort = con.execute("""
    WITH order_flag AS (
        SELECT o.order_id, c.customer_unique_id, o.order_purchase_timestamp,
            CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 'late' ELSE 'on_time' END AS delivery_flag
        FROM orders o JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    ),
    first_orders AS (
        SELECT customer_unique_id, delivery_flag,
            ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY order_purchase_timestamp) AS rn,
            COUNT(*) OVER (PARTITION BY customer_unique_id) AS n_total_orders
        FROM order_flag
    )
    SELECT delivery_flag, COUNT(*) AS n_first_time_customers,
           SUM(CASE WHEN n_total_orders>1 THEN 1 ELSE 0 END) AS became_repeat,
           ROUND(100.0*SUM(CASE WHEN n_total_orders>1 THEN 1 ELSE 0 END)/COUNT(*),2) AS repeat_rate_pct
    FROM first_orders WHERE rn = 1 GROUP BY 1
""").fetchdf()

cohort.to_csv(EXPORT_DIR / "repeat_purchase_cohort.csv", index=False)
cohort


,delivery_flag,n_first_time_customers,became_repeat,repeat_rate_pct
0,on_time,85745,2609.0,3.04
1,late,7605,192.0,2.52


In [7]:
# Two-proportion z-test on the repeat-rate gap
import math
from scipy import stats

row = cohort.set_index("delivery_flag")
p1_n, p1_x = int(row.loc["on_time","n_first_time_customers"]), int(row.loc["on_time","became_repeat"])
p2_n, p2_x = int(row.loc["late","n_first_time_customers"]), int(row.loc["late","became_repeat"])

p1, p2 = p1_x/p1_n, p2_x/p2_n
pooled = (p1_x+p2_x)/(p1_n+p2_n)
se = math.sqrt(pooled*(1-pooled)*(1/p1_n + 1/p2_n))
z = (p1-p2)/se
pval = 2*(1-stats.norm.cdf(abs(z)))

print(f"on_time repeat rate: {p1:.4%}   late repeat rate: {p2:.4%}   gap: {(p1-p2)*100:.2f}pp")
print(f"z = {z:.2f}, two-tailed p = {pval:.4f}")


on_time repeat rate: 3.0427%   late repeat rate: 2.5247%   gap: 0.52pp
z = 2.54, two-tailed p = 0.0111


## 5. Revenue at risk

In [8]:
incremental = con.execute("""
    WITH cust_orders AS (
        SELECT c.customer_unique_id, o.order_id, o.order_purchase_timestamp,
            ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp) AS rn
        FROM orders o JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'
    ),
    order_value AS (
        SELECT order_id, SUM(price+freight_value) AS order_value FROM order_items GROUP BY 1
    ),
    per_repeat_customer AS (
        SELECT co.customer_unique_id, SUM(ov.order_value) AS total_2nd_plus_revenue
        FROM cust_orders co JOIN order_value ov ON co.order_id = ov.order_id
        WHERE co.rn > 1 GROUP BY 1
    )
    SELECT COUNT(*) AS n_repeat_customers,
           ROUND(AVG(total_2nd_plus_revenue),2) AS avg_total_incremental_revenue
    FROM per_repeat_customer
""").fetchdf()

avg_incremental = float(incremental["avg_total_incremental_revenue"][0])

late_first_time = p2_n
gap_pp = (p1 - p2)
lost_repeat_customers = late_first_time * gap_pp
revenue_at_risk = lost_repeat_customers * avg_incremental

summary = {
    "overall_late_rate_pct": float(baseline.loc[baseline.delivery_flag=="late","pct_of_delivered"].iloc[0]),
    "avg_review_score_on_time": float(review_impact.loc[review_impact.delivery_flag=="on_time","avg_review_score"].iloc[0]),
    "avg_review_score_late": float(review_impact.loc[review_impact.delivery_flag=="late","avg_review_score"].iloc[0]),
    "pct_1_2_star_on_time": float(review_impact.loc[review_impact.delivery_flag=="on_time","pct_1_2_star"].iloc[0]),
    "pct_1_2_star_late": float(review_impact.loc[review_impact.delivery_flag=="late","pct_1_2_star"].iloc[0]),
    "repeat_rate_on_time_pct": round(p1*100, 2),
    "repeat_rate_late_pct": round(p2*100, 2),
    "repeat_rate_gap_pvalue": round(pval, 4),
    "avg_incremental_revenue_per_repeat_customer": avg_incremental,
    "estimated_lost_repeat_customers": round(lost_repeat_customers, 1),
    "estimated_revenue_at_risk_usd": round(revenue_at_risk, 0),
    "same_state_late_pct": float(late_by_route.loc[late_by_route.route=="same_state","pct_late"].iloc[0]),
    "different_state_late_pct": float(late_by_route.loc[late_by_route.route=="different_state","pct_late"].iloc[0]),
}

with open(EXPORT_DIR / "summary_metrics.json", "w") as f:
    json.dump(summary, f, indent=2)

summary


{'overall_late_rate_pct': 8.11,
 'avg_review_score_on_time': 4.29,
 'avg_review_score_late': 2.57,
 'pct_1_2_star_on_time': 9.23,
 'pct_1_2_star_late': 54.03,
 'repeat_rate_on_time_pct': 3.04,
 'repeat_rate_late_pct': 2.52,
 'repeat_rate_gap_pvalue': np.float64(0.0111),
 'avg_incremental_revenue_per_repeat_customer': 163.15,
 'estimated_lost_repeat_customers': 39.4,
 'estimated_revenue_at_risk_usd': 6428.0,
 'same_state_late_pct': 6.06,
 'different_state_late_pct': 9.27}

## Caveat

This revenue-at-risk figure only captures the *direct* effect of a late order on that same customer's future spend. It does not (and cannot, from this data) capture word-of-mouth or public 1-star reviews deterring *other* prospective customers — the true cost is almost certainly larger. Stated plainly in the memo rather than inflated to compensate.